# Novelty evidence from the recorded robot data (Colab)

Runs **now**, before the RL training: every claim of the framework is answered by a measurement from your own logs,
with *n* and a test, and is stated against the assumption it challenges. Upload a zip of your whole `mycobot_framework`
folder (with `novelty_evidence.py`, `kinematics.py`, `ik.py`, `deploy/`, `hw_logs/`, `results_views/`).

| claim | question | data |
|---|---|---|
| C1 | is the commanded view the obtained view; can the encoders predict the error? | `hw_logs/characterize/*/repeat.csv`, all `moves.csv` |
| C2 | does image verification close the gap? | coverage runs + recorded library (`deploy/views/*/meta.json`) |
| C3 | how deep can the camera go and still be verified? | same, by descent |
| C4 | does the reachability lattice matter vs naive IK? | `deploy/refcover_lattice*.npz` |
| C5 | proposed planner vs ablation vs hand-programmed raster | `results_views/refcover_sweep*.json` |
| C6 | is there room for learning (best heuristic vs oracle, survey predictability)? | `data/view_library.npz` (after collection) |

The wording of each claim follows the data: where a result does **not** support a claim, the table says so.

In [ ]:
# 1) Upload the framework zip
import glob, os, zipfile, sys
if not glob.glob('/content/*.zip'):
    from google.colab import files
    files.upload()
z = sorted(glob.glob('/content/*.zip'), key=os.path.getsize)[-1]
zipfile.ZipFile(z).extractall('/content/fw')
roots = sorted(glob.glob('/content/fw/**/deploy/locked_pose.json', recursive=True), key=len)
assert roots, 'deploy/locked_pose.json not found: zip the whole mycobot_framework folder'
ROOT = os.path.dirname(os.path.dirname(roots[0])); os.chdir(ROOT); sys.path.insert(0, ROOT)
print('framework folder:', ROOT)
for f in ['novelty_evidence.py', 'kinematics.py', 'ik.py', 'views.py']:
    print('  %-22s %s' % (f, 'ok' if os.path.exists(f) else 'MISSING'))
print('  coverage runs:', len(glob.glob('hw_logs/refcover*/*/views.csv')), '| library placements:',
      len(glob.glob('deploy/views/*/meta.json')), '| characterization sessions:', len(glob.glob('hw_logs/characterize/*/repeat.csv')),
      '| lattices:', [os.path.basename(f) for f in glob.glob('deploy/refcover_lattice*.npz')],
      '| sweeps:', [os.path.basename(f) for f in glob.glob('results_views/refcover_sweep*.json')])

In [ ]:
# 2) Compute all claims (C4 re-solves 250 lattice nodes with naive IK: ~1-5 min)
import importlib, novelty_evidence as NE
importlib.reload(NE); NE.CLAIMS.clear()
for fn, args in ((NE.c1_encoders, ()), (NE.c2_c3_verification, ()), (NE.c4_lattice, (250,)), (NE.c5_planner, ()),
                 (NE.c6_learning_room, ())):
    try:
        fn(*args)
    except Exception as e:
        print('  %s failed: %s' % (fn.__name__, e))

In [ ]:
# 3) Claims table (copy into the paper's results / response to reviewers) and figures
import pandas as pd, csv
from IPython.display import display, Image, Markdown
T = pd.DataFrame(NE.CLAIMS); os.makedirs(NE.OUT, exist_ok=True)
T.to_csv(os.path.join(NE.OUT, 'claims_table.csv'), index=False)
pd.set_option('display.max_colwidth', 120); display(T)
for f in sorted(glob.glob(os.path.join(NE.OUT, '*.png'))):
    display(Markdown('**' + os.path.basename(f) + '**')); display(Image(f, width=750))

In [ ]:
# 4) Download figures (PNG + PDF) and the claims table
import shutil
shutil.make_archive('/content/results_novelty', 'zip', ROOT, NE.OUT)
from google.colab import files
files.download('/content/results_novelty.zip')